In [6]:
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import tiktoken

In [7]:
# concatenate the dataframes 'openai_ZS_multiclass1_1.csv' and 'openai_ZS_multiclass1_2.csv'
df1 = pd.read_csv('openai_ZS_multiclass1_1.csv')
df2 = pd.read_csv('openai_ZS_multiclass1_2.csv')

pred_df = pd.concat([df1, df2], ignore_index=True)

pred_df.to_csv('openai_ZS_multiclass1.csv', index=False)

In [8]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Accuracy: 0.722810
F1 score: 0.721135
Precision: 0.728165
Recall: 0.722810
Average response time: 0.6001199126748222
Average completion tokens: 2.0
Average prompt tokens: 112.70856398567241
Average total tokens: 114.70856398567241


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.22241760000000224


In [12]:
with open('openai_ZS_multiclass1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')
    f.write(f'Lines classified: {len(pred_df)}\n')